# Tahoe-100M → PTDS 同细胞系、同药物提取（Google Colab）

目标：从 [Tahoe 官方数据](https://huggingface.co/datasets/tahoebio/Tahoe-100M) 中提取与 **当前 PTDS 样本表**同细胞系、同药物母体的单药 RNA 数据。**不限制剂量，不把不同剂量合并，不限制到 UniPert 的 20 种药或 MCF7。**

- 输入：`01_sample_info.xlsx`；不需要蛋白强度矩阵，也不需要先手动下载 Tahoe 全集。
- 药物：PubChem CID 相同，或 RDKit 标准化母体完整 InChIKey 相同（保留立体化学）。CID 可补充原始 canonical SMILES 丢失立体信息的情况；仅连接骨架相似而立体信息不同的候选进入复核表，不自动通过。复用上一份 UniPert notebook 的 PubChem 缓存（若存在）。
- 细胞系：用官方表映射 Cellosaurus ID。**注释表中的细胞系候选不代表表达数据实际有覆盖**；最终以逐细胞记录为准。
- 下载策略：读取 Parquet footer，用 sample 的 min/max 跳过无关行组；再读取小型元数据列做精确 `(sample, cell_line_id)` 筛选，只有命中时才读取该行组的表达数组。Hugging Face 支持 HTTP Range 是此工作流的前提。
- Tahoe 仍是大数据：官方有 3,388 个分片，即使按索引筛选，也可能需要较长时间和数 GB 至数十 GB 网络流量，**不是只下载最终子集的字节数**。索引结束会显示预计表达列读取量；实际受缓存与压缩影响。
- 默认保存筛选后的单细胞 Parquet 分片，以及按 `sample + cell_line_id + plate` 求和的 **pseudobulk 原始 counts**。默认 4 个分片并发，输出先在 Colab 本地压缩，再复制到 Drive；每个源分片单独提交 checkpoint。
- 匹配对照：保留同细胞系、相关 plate 的 `DMSO_TF`；不跨 plate 自动补对照。样本级汇总不是新增生物学重复。
- 输出：`MyDrive/virtualcell_ptds_tahoe_alignment_v1/`。单细胞结果可能较大，请保证 Drive 有空间。
- `MAX_SHARDS=None` 表示扫描全部。若设置为例如 10 试跑，输出会明确标为 **partial**，不能据此断言某个条件不存在。

来源：[官方数据说明](https://huggingface.co/datasets/tahoebio/Tahoe-100M/blob/main/README.md)、[官方读取教程](https://huggingface.co/datasets/tahoebio/Tahoe-100M/blob/main/tutorials/loading_data.ipynb)。不使用已迁移的旧 Arc GCS 地址，不需要 Google Cloud 计费项目。


In [ ]:
%pip -q install "pandas>=2,<3" "numpy>=1.26,<3" "anndata>=0.11,<0.13" pyarrow scipy requests openpyxl rdkit tqdm


## 1. Drive、输入与运行配置
上传样本表或修改 Drive 路径。CPU 即可。保持 `MAX_SHARDS=None` 才是完整扫描；`SAVE_SINGLE_CELL=False` 可只保存 pseudobulk 和对应元数据，减少 Drive 占用。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import io, json, re, ast, time, hashlib, shutil, threading, bisect, tempfile
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import anndata as ad
from scipy import sparse
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from rdkit import Chem, rdBase
from rdkit.Chem.MolStandardize import rdMolStandardize
from tqdm.auto import tqdm
from IPython.display import display

BASE=Path('/content/drive/MyDrive/virtualcell_ptds_tahoe_alignment_v1')
PTDS_XLSX=Path('/content/drive/MyDrive/CrossOmics-Generalization/baseline/proteintalks/01_sample_info.xlsx')
PREVIOUS_CACHE=Path('/content/drive/MyDrive/virtualcell_ptds_unipert_alignment_v1/raw/pubchem_cache.json')
REPO='tahoebio/Tahoe-100M'
MAX_SHARDS=None  # 全量扫描；整数仅试跑，结果标为 partial
INDEX_WORKERS=4
EXTRACT_WORKERS=4  # 同时提取的分片数；可试 6，内存紧张或频繁 429 时降至 2
LOCAL_SCRATCH=Path('/content/tahoe_ptds_scratch')  # 临时压缩文件，不保存整个源分片
SAVE_SINGLE_CELL=True
RAW, OUT=BASE/'raw', BASE/'results'
for p in (RAW,OUT): p.mkdir(parents=True,exist_ok=True)
assert Path('/content/drive/MyDrive').is_dir(), '先挂载 Drive'
if not PTDS_XLSX.exists():
    from google.colab import files
    uploaded=files.upload()
    assert '01_sample_info.xlsx' in uploaded
    PTDS_XLSX=RAW/'01_sample_info.xlsx'
    PTDS_XLSX.write_bytes(uploaded['01_sample_info.xlsx'])
if PREVIOUS_CACHE.exists() and not (RAW/'pubchem_cache.json').exists():
    shutil.copyfile(PREVIOUS_CACHE,RAW/'pubchem_cache.json')
_sessions=threading.local()
def http_session():
    if not hasattr(_sessions,'value'):
        s=requests.Session()
        s.mount('https://',HTTPAdapter(max_retries=Retry(total=4,backoff_factor=1,
            status_forcelist=[429,500,502,503,504],allowed_methods=['GET'])))
        s.headers['User-Agent']='PTDS-Tahoe-extraction/1.0'
        _sessions.value=s
    return _sessions.value
session=http_session()
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''): h.update(b)
    return h.hexdigest()
def atomic_json(p,obj):
    p=Path(p);t=p.with_suffix(p.suffix+'.part')
    t.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8');t.replace(p)
def save_csv(df,name): df.to_csv(OUT/name,index=False,encoding='utf-8-sig')
def cell_key(x): return re.sub('[^A-Z0-9]','',str(x).upper())
def drug_key(x): return str(x).strip().casefold()
pin=RAW/'source_revision.json'
if pin.exists(): source=json.loads(pin.read_text())
else:
    r=session.get(f'https://huggingface.co/api/datasets/{REPO}',timeout=60);r.raise_for_status()
    source={'repo':REPO,'revision':r.json()['sha']};atomic_json(pin,source)
REVISION=source['revision']
assert source['repo']==REPO
def source_url(path): return f'https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{path}'
def fetch_metadata(name):
    dest=RAW/(REVISION+'_'+name+'.parquet')
    receipt=dest.with_suffix('.sha256')
    if dest.exists() and receipt.exists() and receipt.read_text()==sha256(dest): return pd.read_parquet(dest)
    temp=dest.with_suffix('.part')
    with session.get(source_url('metadata/'+name+'.parquet'),stream=True,timeout=(30,180)) as r:
        r.raise_for_status()
        with temp.open('wb') as f:
            for b in r.iter_content(1024**2): f.write(b)
    temp.replace(dest);receipt.write_text(sha256(dest));return pd.read_parquet(dest)
print('固定数据版本:',REVISION,'输出:',OUT)


## 2. PTDS 候选细胞系与药物结构
只读取 PTDS 单药处理。保留全部剂量、时间、重复及原药名；`no` 不当作药物。
细胞系名称只规范大小写、空格和标点，不把来源不同的亚系强行合并。需要特殊别名时编辑 `CELL_ALIASES`。


In [ ]:
ptds=pd.read_excel(PTDS_XLSX,dtype={'Sample_ID':str})
assert {'Sample_ID','Cell_Line','Drug_Type','Pert_Name1','Pert_Name2'}<=set(ptds)
protein=ptds.loc[ptds.Drug_Type.eq('single')].copy()
protein=protein.loc[~protein.Pert_Name1.fillna('').map(drug_key).isin(['','no','ctrl','control','dmso','vehicle'])].copy()
assert protein.Pert_Name2.fillna('').astype(str).str.strip().eq('').all(), 'single 行有第二药物，需复核'
protein['cell_key']=protein.Cell_Line.map(cell_key)
CELL_ALIASES={}  # 仅加入已经核验的别名，例如 {'原名':'官方名称'}
protein['cell_key']=protein.Cell_Line.map(lambda s:cell_key(CELL_ALIASES.get(s,s)))
cells_meta=fetch_metadata('cell_line_metadata')
drugs_meta=fetch_metadata('drug_metadata')
samples_meta=fetch_metadata('sample_metadata')
genes_meta=fetch_metadata('gene_metadata')
cellmap=cells_meta[['cell_name','Cell_ID_Cellosaur']].drop_duplicates().copy()
cellmap['cell_key']=cellmap.cell_name.map(cell_key)
assert cellmap.groupby('cell_key').Cell_ID_Cellosaur.nunique().le(1).all()
pcells=protein[['Cell_Line','cell_key']].drop_duplicates().merge(cellmap,on='cell_key',how='left',validate='many_to_one')
save_csv(pcells,'ptds_cell_candidates.csv')
display(pcells)
protein=protein.merge(cellmap[['cell_key','Cell_ID_Cellosaur']],on='cell_key',how='left',validate='many_to_one')
print('候选不是实际覆盖；完整扫描结束后才能确认。')


In [ ]:
from urllib.parse import quote
def parent_key(mol):
    if mol is None: return None
    try:
        mol=rdMolStandardize.Cleanup(mol)
        mol=rdMolStandardize.FragmentParent(mol)
        mol=rdMolStandardize.Uncharger().uncharge(mol)
        return Chem.MolToInchiKey(mol) or None
    except Exception: return None

cache_path=RAW/'pubchem_cache.json'
cache=json.loads(cache_path.read_text()) if cache_path.exists() else {}
override_path=BASE/'drug_overrides.csv'
overrides=pd.read_csv(override_path) if override_path.exists() else pd.DataFrame(columns=['ptds_drug','pubchem_cid','reason'])
assert not overrides.ptds_drug.duplicated().any(), '同一药物只能有一个人工 CID'
def resolve_drug(name):
    query=re.sub(r'\s*\(=#\d+\)', '', str(name)).strip()
    manual=overrides.loc[overrides.ptds_drug.eq(name)]
    if len(manual):
        row=manual.iloc[0]
        assert pd.notna(row.reason) and str(row.reason).strip(), '人工映射必须记录依据'
        endpoint=f'cid/{int(row.pubchem_cid)}'
        basis='manual_cid: '+str(row.reason)
    else:
        endpoint='name/'+quote(query,safe='')
        basis='pubchem_name_unique_candidate'
    url=f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{endpoint}/property/InChI,InChIKey/JSON'
    try:
        if url in cache: props=cache[url]
        else:
            r=session.get(url,timeout=(30,60))
            r.raise_for_status()
            props=r.json()['PropertyTable']['Properties']
            cache[url]=props
            cache_path.write_text(json.dumps(cache,ensure_ascii=False,indent=2),encoding='utf-8')
            time.sleep(0.25)
        if len(props)!=1: return {'ptds_drug':name,'query':query,'status':'ambiguous','basis':basis,'parent_inchikey':None}
        p=props[0]
        key=parent_key(Chem.MolFromInchi(p['InChI']))
        return {'ptds_drug':name,'query':query,'pubchem_cid':p['CID'],'pubchem_inchikey':p['InChIKey'],
                'parent_inchikey':key,'status':'resolved' if key else 'structure_failed','basis':basis,'query_url':url}
    except Exception as e:
        return {'ptds_drug':name,'query':query,'status':'lookup_failed','basis':basis,'parent_inchikey':None,'error':str(e)}


drugmap=pd.DataFrame([resolve_drug(n) for n in sorted(protein.Pert_Name1.unique())])
save_csv(drugmap,'ptds_drug_resolution.csv')
save_csv(drugmap.loc[drugmap.status.ne('resolved')],'drug_resolution_review.csv')
assert drugmap.status.eq('resolved').any(), '全部药物解析失败，请检查网络或 drug_overrides.csv；不能把查询失败当成零交集'
drugs_meta['parent_inchikey']=drugs_meta.canonical_smiles.map(
    lambda x:parent_key(Chem.MolFromSmiles(x)) if isinstance(x,str) else None)
drugs_meta['drug_key']=drugs_meta.drug.map(drug_key)
save_csv(drugs_meta,'tahoe_drug_structure_map.csv')
display(drugmap.reindex(columns=['ptds_drug','status','pubchem_cid','parent_inchikey']))
print('未解析/歧义药物:',int(drugmap.status.ne('resolved').sum()))


## 3. 建立目标 (sample, cell_line_id) 与同 plate 对照
`drugname_drugconc` 通过安全的 literal_eval 解析，剂量不用于筛选。多药条件不自动拆成单药。
若元数据结构变化则报错，避免给未知剂量填默认数值。论文主实验 24 h，仅作为论文层面的注释保留。


In [ ]:
def parse_condition(value):
    v=ast.literal_eval(value) if isinstance(value,str) else value
    if not isinstance(v,(list,tuple)) or not len(v): raise ValueError(f'无法解析条件: {value}')
    if len(v)!=1: return {'single_compound':False,'dose':None,'dose_unit':None}
    row=v[0]
    if not isinstance(row,(list,tuple)) or len(row)!=3: raise ValueError(f'条件格式变化: {value}')
    return {'single_compound':True,'dose':float(row[1]),'dose_unit':str(row[2])}
assert samples_meta['sample'].is_unique
samples_meta=pd.concat([samples_meta.reset_index(drop=True),pd.DataFrame([parse_condition(x) for x in samples_meta.drugname_drugconc])],axis=1)
samples_meta['drug_key']=samples_meta.drug.map(drug_key)
assert drugs_meta.groupby('drug_key').parent_inchikey.nunique().le(1).all(), '标准化药名对应多个结构，需复核'
sm=samples_meta.merge(drugs_meta[['drug_key','parent_inchikey']].drop_duplicates('drug_key'),on='drug_key',how='left',validate='many_to_one')
ptpairs=protein.dropna(subset=['Cell_ID_Cellosaur'])[['Cell_Line','Cell_ID_Cellosaur','Pert_Name1']].drop_duplicates()
ptpairs=ptpairs.merge(drugmap.loc[drugmap.status.eq('resolved'),['ptds_drug','parent_inchikey']],left_on='Pert_Name1',right_on='ptds_drug',how='inner')
def cid_key(v):
    try:
        x=float(v)
        return str(int(x)) if np.isfinite(x) and x>0 and x.is_integer() else None
    except (TypeError,ValueError):return None
pmap=drugmap.loc[drugmap.status.eq('resolved')].copy()
pmap['cid_key']=pmap.pubchem_cid.map(cid_key)
drugs_meta['cid_key']=drugs_meta.pubchem_cid.map(cid_key)
links=[]
for key,basis in [('cid_key','exact_pubchem_cid'),('parent_inchikey','standardized_parent_full_inchikey')]:
    linked=pmap.dropna(subset=[key]).merge(drugs_meta.dropna(subset=[key]),on=key,how='inner')
    links.append(linked[['ptds_drug','drug_key']].assign(match_basis=basis))
druglinks=pd.concat(links,ignore_index=True).groupby(['ptds_drug','drug_key'],as_index=False).match_basis.agg(lambda v:';'.join(sorted(set(v))))
save_csv(druglinks,'drug_match_evidence.csv')
conditions=ptpairs.merge(druglinks,on='ptds_drug',how='inner').merge(
    sm.loc[sm.single_compound].drop(columns=['parent_inchikey']),on='drug_key',how='inner')
# 完整 InChIKey 不同但连接骨架相同的情况保留供人工判断，不能当作同一立体异构体。
pmap['connectivity']=pmap.parent_inchikey.str[:14]
drugs_meta['connectivity']=drugs_meta.parent_inchikey.str[:14]
review=pmap.dropna(subset=['connectivity']).merge(drugs_meta.dropna(subset=['connectivity']),on='connectivity',suffixes=('_ptds','_tahoe'))
review=review.loc[review.parent_inchikey_ptds.ne(review.parent_inchikey_tahoe)]
accepted=set(zip(druglinks.ptds_drug,druglinks.drug_key))
review=review.loc[[ (n,k) not in accepted for n,k in zip(review.ptds_drug,review.drug_key) ]]
save_csv(review,'structure_match_review.csv')
coverage=drugmap[['ptds_drug','status']].copy()
coverage['n_tahoe_drug_annotations']=coverage.ptds_drug.map(druglinks.groupby('ptds_drug').drug_key.nunique()).fillna(0).astype(int)
save_csv(coverage,'drug_candidate_coverage.csv')
conditions=conditions.rename(columns={'Cell_ID_Cellosaur':'cell_line_id'})
save_csv(conditions,'candidate_conditions.csv')
assert len(conditions), '没有元数据候选；先检查药物复核表和细胞系候选'
selected_plates=set(conditions.plate)
control_samples=sm.loc[sm.drug_key.eq('dmso_tf') & sm.plate.isin(selected_plates)]
targets=conditions[['sample','cell_line_id','plate']].drop_duplicates().assign(alignment_role='matched_drug')
# 只加本 plate 中确有 PTDS 药物候选的细胞系。
controls=conditions[['cell_line_id','plate']].drop_duplicates().merge(
    control_samples[['sample','plate']],on='plate').assign(alignment_role='plate_control')
targets=pd.concat([targets,controls],ignore_index=True).drop_duplicates(['sample','cell_line_id'])
target_keys=set(zip(targets['sample'],targets.cell_line_id))
target_samples=sorted(set(targets['sample']))
assert not targets.duplicated(['sample','cell_line_id']).any()
save_csv(targets,'target_sample_cells.csv')
scope={'repo':REPO,'revision':REVISION,'ptds_sha256':sha256(PTDS_XLSX),
       'targets':sorted([list(k) for k in target_keys]),'save_single_cell':SAVE_SINGLE_CELL,
       'rdkit_version':rdBase.rdkitVersion,'parser':'tahoe-cls-counts-v2-cid-and-structure'}
SCOPE=hashlib.sha256(json.dumps(scope,sort_keys=True).encode()).hexdigest()[:16]
WORK=BASE/'checkpoints'/SCOPE
INDEX=BASE/'footer_index'/REVISION
PARTS=OUT/('parts_'+SCOPE)
for p in (WORK,INDEX,PARTS):p.mkdir(parents=True,exist_ok=True)
atomic_json(WORK/'config.json',scope)
atomic_json(OUT/'scan_status.json',{'scope':SCOPE,'complete':False,'status':'running'})
atomic_json(OUT/'summary.json',{'scope':SCOPE,'complete':False,'status':'running'})
print('候选细胞系:',sorted(targets.cell_line_id.unique()),'目标样本数:',len(target_samples),'运行标识:',SCOPE)


## 4. 索引分片：仅读取 footer，不先下载全量表达
扫描列表固定到同一 commit。每个成功读取的 footer 保存统计索引；失败项不当成空结果，下次重跑会重试。
`MAX_SHARDS` 是源分片扫描上限，**不是匹配细胞数上限**。可先设 10 试跑，再恢复 None 重跑本节及后续。


In [ ]:
class HTTPRange(io.RawIOBase):
    def __init__(self,url,size):
        self.url,self.size,self.pos=url,int(size),0
        self.cache=[]
        self.download_url=url
        self.reuse_download_url=True
    def readable(self):return True
    def seekable(self):return True
    def tell(self):return self.pos
    def seek(self,offset,whence=0):
        self.pos=offset if whence==0 else (self.pos if whence==1 else self.size)+offset
        if self.pos<0:raise ValueError('Negative seek')
        return self.pos
    def read(self,n=-1):
        if globals().get('_EXTRACTION_STOP') is not None and _EXTRACTION_STOP.is_set():
            raise InterruptedError('提取已中断')
        n=min(self.size-self.pos,n if n>=0 else self.size-self.pos)
        if n<=0:return b''
        for start,data in reversed(self.cache):
            if start<=self.pos and self.pos+n<=start+len(data):
                result=data[self.pos-start:self.pos-start+n];self.pos+=len(result);return result
        start=(self.pos//65536)*65536 if n<=131072 else self.pos
        end=min(self.size-1,((self.pos+n+65535)//65536)*65536-1) if n<=131072 else self.pos+n-1
        wanted=f'bytes {start}-{end}/{self.size}'
        target=self.download_url if self.reuse_download_url else self.url
        for attempt in range(2):
            with http_session().get(target,headers={'Range':f'bytes={start}-{end}'},
                                    stream=True,timeout=(15,60)) as r:
                # 某些 CDN 的签名仅适用于原始范围；此时退回 resolve 地址。
                if target!=self.url and (r.status_code in [401,403] or
                    (r.status_code in [200,206] and r.headers.get('Content-Range')!=wanted)):
                    self.reuse_download_url=False;self.download_url=self.url;target=self.url
                    continue
                if r.status_code!=206:raise IOError(f'需要 HTTP 206，收到 {r.status_code}；不自动全量下载')
                if r.headers.get('Content-Range')!=wanted:raise IOError('服务器返回了错误的 Range')
                if self.reuse_download_url:self.download_url=r.url
                data=b''.join(r.iter_content(1024**2))
                break
        else:raise IOError('Range 地址刷新失败')
        if len(data)!=end-start+1:raise IOError('不完整的 Range 响应')
        if n<=131072:self.cache=(self.cache+[(start,data)])[-4:]
        result=data[self.pos-start:self.pos-start+n]
        self.pos+=len(result);return result

if globals().get('_EXTRACTION_STOP') is not None:_EXTRACTION_STOP.clear()
inventory_path=RAW/(REVISION+'_shards.json')
if inventory_path.exists():shards=json.loads(inventory_path.read_text())
else:
    url=f'https://huggingface.co/api/datasets/{REPO}/tree/{REVISION}/data?limit=1000'
    shards=[]
    while url:
        r=session.get(url,timeout=90);r.raise_for_status()
        shards.extend({'path':x['path'],'size':x['size']} for x in r.json()
                      if x['type']=='file' and x['path'].endswith('.parquet'))
        url=r.links.get('next',{}).get('url')
    shards=sorted(shards,key=lambda x:x['path'])
    assert shards, '没有找到数据分片'
    atomic_json(inventory_path,shards)
if MAX_SHARDS is not None:assert isinstance(MAX_SHARDS,int) and MAX_SHARDS>0
run_shards=shards if MAX_SHARDS is None else shards[:MAX_SHARDS]
def footer_index(item):
    name=Path(item['path']).stem
    dest=INDEX/(name+'.json')
    if dest.exists():return json.loads(dest.read_text())
    with HTTPRange(source_url(item['path']),item['size']) as remote:
        pf=pq.ParquetFile(remote)
        required={'sample','cell_line_id','plate','genes','expressions','drug','BARCODE_SUB_LIB_ID'}
        assert required<=set(pf.schema_arrow.names), '数据列发生变化'
        groups=[]
        for i in range(pf.num_row_groups):
            rg=pf.metadata.row_group(i);stats={};count_bytes=0
            for j in range(rg.num_columns):
                col=rg.column(j)
                if col.path_in_schema in ['sample','plate']:
                    st=col.statistics
                    stats[col.path_in_schema]=[str(st.min),str(st.max)] if st and st.has_min_max else None
                if col.path_in_schema.startswith(('genes.','expressions.')):count_bytes+=col.total_compressed_size
            groups.append({'row_group':i,'rows':rg.num_rows,'stats':stats,'expression_compressed_bytes':count_bytes})
        result={**item,'rows':pf.metadata.num_rows,'groups':groups}
    atomic_json(dest,result);return result
def could_contain(bounds,values):
    if bounds is None:return True
    i=bisect.bisect_left(values,bounds[0])
    return i<len(values) and values[i]<=bounds[1]
plans=[];index_errors=[]
with ThreadPoolExecutor(max_workers=INDEX_WORKERS) as pool:
    futures={pool.submit(footer_index,x):x for x in run_shards}
    for fut in tqdm(as_completed(futures),total=len(futures),desc='Parquet 索引'):
        item=futures[fut]
        try:
            idx=fut.result()
            keep=[g for g in idx['groups'] if could_contain(g['stats'].get('sample'),target_samples)
                  and could_contain(g['stats'].get('plate'),sorted(selected_plates))]
            if keep:plans.append({**item,'groups':keep})
        except Exception as e:index_errors.append({'path':item['path'],'error':str(e)})
plans.sort(key=lambda x:x['path'])
atomic_json(WORK/'plan.json',plans)
atomic_json(OUT/'index_errors.json',index_errors)
estimate=sum(g['expression_compressed_bytes'] for x in plans for g in x['groups'])
print('全部分片:',len(shards),'本轮扫描:',len(run_shards),'候选分片:',len(plans),'索引失败:',len(index_errors))
print('候选行组表达列读取量上界估计 GB:',round(estimate/1e9,2),'（另有 footer 和元数据流量）')


## 5. 分片提取与 checkpoint
每个行组先筛元数据，命中后读取表达数组。移除官方 CLS 标记（首项负表达值），对剩余 counts、token 和细胞 ID 做检查。
使用 `EXTRACT_WORKERS` 控制有限并发，不一次排队数千个任务。压缩和哈希在本地临时目录完成，复制到 Drive 并核对大小后提交 checkpoint；续跑仍逐文件核验 SHA-256。
中断时取消未开始任务，并等待正在运行的请求退出；看到“提取线程已停止”后再重跑本节，避免两个运行同时写相同文件。完成顺序可能不同，最终按源分片路径排序。


In [ ]:
assert genes_meta.token_id.is_unique and genes_meta.ensembl_id.is_unique
genes_meta=genes_meta.sort_values('token_id').reset_index(drop=True)
token_to_col={int(t):i for i,t in enumerate(genes_meta.token_id)}
n_genes=len(genes_meta)
_EXTRACTION_STOP=threading.Event()
def clean_counts(tokens,counts):
    tokens=list(tokens);counts=list(counts)
    if len(tokens)!=len(counts):raise ValueError('token 与 count 长度不一致')
    if counts and counts[0]<0:tokens=tokens[1:];counts=counts[1:]
    vals=np.asarray(counts,dtype=np.float64)
    if not np.isfinite(vals).all() or (vals<0).any() or not np.equal(vals,np.floor(vals)).all():
        raise ValueError('不是非负整数原始 counts')
    if any(int(t) not in token_to_col for t in tokens):raise ValueError('遇到未知 gene token')
    return [int(t) for t in tokens],[int(x) for x in counts]
def rows_to_csr(rows):
    data=[];cols=[];indptr=[0]
    for row in rows:
        g,v=clean_counts(row['genes'],row['expressions'])
        row['genes'],row['expressions']=g,v
        data.extend(v);cols.extend(token_to_col[t] for t in g);indptr.append(len(data))
    mat=sparse.csr_matrix((np.asarray(data,dtype=np.int64),np.asarray(cols,dtype=np.int64),np.asarray(indptr,dtype=np.int64)),shape=(len(rows),n_genes))
    mat.sum_duplicates();mat.eliminate_zeros();return mat
def valid_done(item):
    mark=WORK/(Path(item['path']).stem+'.done.json')
    if not mark.exists():return None
    result=json.loads(mark.read_text())
    if result.get('scope')!=SCOPE:return None
    for rec in result['files']:
        p=PARTS/rec['name']
        if not p.exists() or sha256(p)!=rec['sha256']:return None
    return result
def process_shard(item):
    if _EXTRACTION_STOP.is_set():raise InterruptedError('提取已中断')
    prior=valid_done(item)
    if prior is not None:return prior
    stem=Path(item['path']).stem;rows=[]
    with HTTPRange(source_url(item['path']),item['size']) as remote:
        pf=pq.ParquetFile(remote)
        for g in item['groups']:
            if _EXTRACTION_STOP.is_set():raise InterruptedError('提取已中断')
            ri=g['row_group']
            meta=pf.read_row_group(ri,columns=['sample','cell_line_id','plate','drug','BARCODE_SUB_LIB_ID'],use_threads=False).to_pandas()
            mask=np.fromiter(((s,c) in target_keys for s,c in zip(meta['sample'],meta.cell_line_id)),dtype=bool,count=len(meta))
            if not mask.any():continue
            expr=pf.read_row_group(ri,columns=['genes','expressions'],use_threads=False)
            table=pa.Table.from_pandas(meta,preserve_index=False)
            table=table.append_column('genes',expr['genes']).append_column('expressions',expr['expressions'])
            selected=table.filter(pa.array(mask)).to_pylist()
            # sample/plate 必须与小型元数据一致，避免跨 plate 对照。
            expected=samples_meta.set_index('sample')['plate'].to_dict()
            assert all(str(r['plate'])==str(expected[r['sample']]) for r in selected)
            rows.extend(selected)
    files_out=[]
    scratch=Path(globals().get('LOCAL_SCRATCH',Path(tempfile.gettempdir())/'tahoe_ptds_scratch'))
    scratch.mkdir(parents=True,exist_ok=True)
    def commit_file(temp,dest):
        if _EXTRACTION_STOP.is_set():raise InterruptedError('提取已中断')
        digest=sha256(temp)
        staged=dest.with_suffix(dest.suffix+'.part')
        shutil.copyfile(temp,staged)
        if staged.stat().st_size!=temp.stat().st_size:raise IOError('Drive 文件复制不完整')
        staged.replace(dest)
        files_out.append({'name':dest.name,'sha256':digest})
        temp.unlink(missing_ok=True)
    group_records=[]
    if rows:
        identifiers=[r['BARCODE_SUB_LIB_ID'] for r in rows]
        assert len(set(identifiers))==len(identifiers), '分片内细胞 ID 重复'
        X=rows_to_csr(rows)
        meta=pd.DataFrame([{k:v for k,v in r.items() if k not in ['genes','expressions']} for r in rows])
        key_cols=['sample','cell_line_id','plate']
        keys=pd.MultiIndex.from_frame(meta[key_cols])
        codes,unique=pd.factorize(keys,sort=True)
        group=sparse.csr_matrix((np.ones(len(rows),dtype=np.int64),(codes,np.arange(len(rows)))),shape=(len(unique),len(rows)))
        pb=group@X
        pbobs=unique.to_frame(index=False);pbobs.columns=key_cols
        pbobs['n_cells']=np.bincount(codes,minlength=len(unique))
        assert int(pb.sum())==int(X.sum())
        for kind in ['npz','csv']:
            dest=PARTS/(stem+'.pb.'+kind)
            temp=scratch/dest.name
            if kind=='npz':sparse.save_npz(temp,pb)
            else:pbobs.to_csv(temp,index=False)
            commit_file(temp,dest)
        # 每个细胞的元数据用于追溯；原始 counts 存在可选 Parquet 分片中。
        dest=PARTS/(stem+'.cells.parquet');temp=scratch/dest.name
        pq.write_table(pa.Table.from_pandas(meta,preserve_index=False),temp,compression='zstd');commit_file(temp,dest)
        if SAVE_SINGLE_CELL:
            dest=PARTS/(stem+'.counts.parquet');temp=scratch/dest.name
            pq.write_table(pa.Table.from_pylist(rows),temp,compression='zstd');commit_file(temp,dest)
        group_records=pbobs.to_dict('records')
    result={'scope':SCOPE,'path':item['path'],'n_cells':len(rows),'files':files_out,'groups':group_records}
    atomic_json(WORK/(stem+'.done.json'),result)
    return result

def extract_concurrently(items,workers):
    assert isinstance(workers,int) and 1<=workers<=8, 'EXTRACT_WORKERS 请设为 1 到 8'
    assert len({x['path'] for x in items})==len(items), '重复分片不能并发写入'
    _EXTRACTION_STOP.clear()
    atomic_json(OUT/'scan_status.json',{'scope':SCOPE,'complete':False,'status':'extracting',
        'extraction_workers':workers,'candidate_shards':len(items)})
    results=[];errors=[];pending={};iterator=iter(items)
    pool=ThreadPoolExecutor(max_workers=workers)
    progress=tqdm(total=len(items),desc=f'提取分片（{workers} 并发，可续跑）',unit='分片')
    def submit_next():
        item=next(iterator,None)
        if item is not None:pending[pool.submit(process_shard,item)]=item
    try:
        for _ in range(workers):submit_next()
        while pending:
            finished,_=wait(pending,return_when=FIRST_COMPLETED)
            for future in finished:
                item=pending.pop(future)
                try:results.append(future.result())
                except Exception as e:
                    errors.append({'path':item['path'],'error':str(e)})
                    atomic_json(OUT/'extraction_errors.json',errors)
                progress.update(1)
                progress.set_postfix(done=len(results),failed=len(errors))
                submit_next()
    except KeyboardInterrupt:
        _EXTRACTION_STOP.set()
        for future in pending:future.cancel()
        atomic_json(OUT/'scan_status.json',{'scope':SCOPE,'complete':False,'status':'interrupted',
            'completed_in_this_invocation':len(results),'note':'已提交的分片保留；等待提取线程停止后再重跑'})
        print('正在停止请求，请等待提取线程退出后再重跑。',flush=True)
        raise
    finally:
        pool.shutdown(wait=True,cancel_futures=True)
        progress.close()
        if _EXTRACTION_STOP.is_set():print('提取线程已停止，可以重跑本节。',flush=True)
    return sorted(results,key=lambda x:x['path']),errors

print('提取并发数:',globals().get('EXTRACT_WORKERS',4),'；匹配范围与原版一致。')
completed,extraction_errors=extract_concurrently(plans,globals().get('EXTRACT_WORKERS',4))
atomic_json(OUT/'extraction_errors.json',extraction_errors)
scan_complete=len(run_shards)==len(shards) and not index_errors and not extraction_errors and len(completed)==len(plans)
status={'scope':SCOPE,'complete':False,'scan_complete':scan_complete,
        'status':'extracted_pending_validation' if scan_complete else 'partial',
        'total_source_shards':len(shards),'scanned_shards':len(run_shards),'candidate_shards':len(plans),
        'completed_candidate_shards':len(completed),'index_errors':len(index_errors),'extraction_errors':len(extraction_errors),
        'selected_cells':sum(x['n_cells'] for x in completed),'single_cell_saved':SAVE_SINGLE_CELL,
        'parts_directory':str(PARTS),'revision':REVISION}
atomic_json(OUT/'scan_status.json',status)
print(json.dumps(status,ensure_ascii=False,indent=2))


## 6. 合并 pseudobulk、确认实际交集及对照覆盖
按小批次合并计数，避免同时载入全部单细胞表达。`pseudobulk_raw_counts.h5ad` 保持原始求和 counts，不做 log 或差异计算。
对照匹配表只连接同细胞系、同 plate；缺失对照会单独列出。完整性以 `scan_status.json` 为准，失败分片修复后重跑索引、提取和本节。


In [ ]:
group_frames=[pd.DataFrame(x['groups']) for x in completed if x['groups']]
if group_frames:
    observations=pd.concat(group_frames,ignore_index=True)
    keys=['sample','cell_line_id','plate']
    obs=observations.groupby(keys,as_index=False,sort=True).n_cells.sum()
    key_to_row={tuple(r):i for i,r in enumerate(obs[keys].itertuples(index=False,name=None))}
    matrix=sparse.csr_matrix((len(obs),n_genes),dtype=np.int64)
    parts_with_cells=[x for x in completed if x['groups']]
    for start in tqdm(range(0,len(parts_with_cells),32),desc='合并 pseudobulk'):
        chunk=parts_with_cells[start:start+32];mats=[];destination=[]
        for part in chunk:
            stem=Path(part['path']).stem
            partobs=pd.read_csv(PARTS/(stem+'.pb.csv'))
            mats.append(sparse.load_npz(PARTS/(stem+'.pb.npz')))
            destination.extend(key_to_row[tuple(r)] for r in partobs[keys].itertuples(index=False,name=None))
        stacked=sparse.vstack(mats,format='csr')
        W=sparse.csr_matrix((np.ones(len(destination),dtype=np.int64),(destination,np.arange(len(destination)))),shape=(len(obs),len(destination)))
        matrix=matrix+W@stacked
    obs=obs.merge(targets,on=keys,how='left',validate='one_to_one')
    assert obs.alignment_role.notna().all()
    obs=obs.merge(samples_meta[['sample','drug','drugname_drugconc','dose','dose_unit']],on='sample',how='left',validate='many_to_one')
    obs['time_h_from_paper']=24  # 论文主实验条件，字段名明确来源，不伪装为逐行测量字段。
    obs['cell_name']=obs.cell_line_id.map(cellmap.drop_duplicates('Cell_ID_Cellosaur').set_index('Cell_ID_Cellosaur').cell_name)
    obs.index=pd.Index([f'pb_{i:05d}' for i in range(len(obs))],name='pseudobulk_id')
    var=genes_meta.set_index('ensembl_id').copy();var.index.name='ensembl_id'
    result=ad.AnnData(X=matrix,obs=obs,var=var)
    result.uns['extraction']={'revision':REVISION,'complete':bool(scan_complete),'scope':SCOPE,
        'grouping':'sample + cell_line_id + plate','counts':'raw summed UMI; CLS removed',
        'match':'cell + exact PubChem CID or standardized parent full InChIKey; dose not filtered'}
    temp=OUT/'pseudobulk_raw_counts.part.h5ad';result.write_h5ad(temp,compression='gzip')
    temp.replace(OUT/'pseudobulk_raw_counts.h5ad')
    save_csv(obs.reset_index(),'pseudobulk_obs.csv')
    save_csv(var.reset_index(),'genes.csv')
    actual=conditions.merge(obs.reset_index()[['pseudobulk_id','sample','cell_line_id','n_cells']],on=['sample','cell_line_id'],how='inner')
    save_csv(actual,'matched_conditions.csv')
    actual_pairs=actual[['Cell_Line','cell_line_id','ptds_drug','parent_inchikey','drug']].drop_duplicates()
    save_csv(actual_pairs,'matched_pairs.csv')
    matched_ptds=protein.merge(actual_pairs[['cell_line_id','ptds_drug']].drop_duplicates(),
        left_on=['Cell_ID_Cellosaur','Pert_Name1'],right_on=['cell_line_id','ptds_drug'],how='inner')
    save_csv(matched_ptds,'matched_ptds_samples.csv')
    treated=obs.reset_index().loc[lambda d:d.alignment_role.eq('matched_drug')]
    control=obs.reset_index().loc[lambda d:d.alignment_role.eq('plate_control')]
    cp=treated[['pseudobulk_id','cell_line_id','plate']].merge(
        control[['pseudobulk_id','cell_line_id','plate']],on=['cell_line_id','plate'],how='left',suffixes=('_treated','_control'))
    save_csv(cp,'pseudobulk_control_pairs.csv')
    save_csv(cp.loc[cp.pseudobulk_id_control.isna()],'missing_plate_controls.csv')
    seen=obs[['sample','cell_line_id']].assign(observed=True)
    coverage=targets.merge(seen,on=['sample','cell_line_id'],how='left')
    coverage['coverage_status']=np.where(coverage.observed.eq(True),'observed',
        'not_observed_in_complete_scan' if scan_complete else 'not_observed_in_partial_scan')
    save_csv(coverage,'condition_coverage.csv')
    # 细胞 ID 全局重复检查：按源分片逐个读一列，不读取表达矩阵。
    cellids=set();total=0
    for part in completed:
        if not part['n_cells']:continue
        stem=Path(part['path']).stem
        ids=pq.read_table(PARTS/(stem+'.cells.parquet'),columns=['BARCODE_SUB_LIB_ID']).column(0).to_pylist()
        if any(i in cellids for i in ids):raise ValueError('跨分片细胞 ID 重复，请核对数据版本')
        cellids.update(ids);total+=len(ids)
    assert total==int(obs.n_cells.sum())==status['selected_cells']
    status.update(complete=bool(scan_complete),validation_passed=True,status='complete' if scan_complete else 'partial')
    atomic_json(OUT/'scan_status.json',status)
    summary={**status,'matched_ptds_drugs':int(actual.ptds_drug.nunique()),
        'matched_cell_lines':sorted(actual.cell_line_id.unique().tolist()),'actual_cell_drug_pairs':len(actual_pairs[['cell_line_id','ptds_drug']].drop_duplicates()),
        'pseudobulk_rows':len(obs),'genes':n_genes,'missing_control_groups':int(cp.pseudobulk_id_control.isna().sum()),
        'unresolved_ptds_drugs':int(drugmap.status.ne('resolved').sum()),
        'generated_utc':datetime.now(timezone.utc).isoformat()}
    atomic_json(OUT/'summary.json',summary)
    display(actual_pairs)
    print(json.dumps(summary,ensure_ascii=False,indent=2))
else:
    for name in ['pseudobulk_raw_counts.h5ad','pseudobulk_obs.csv','genes.csv','matched_conditions.csv',
                 'matched_pairs.csv','matched_ptds_samples.csv','pseudobulk_control_pairs.csv',
                 'missing_plate_controls.csv','condition_coverage.csv']:
        (OUT/name).unlink(missing_ok=True)
    status.update(complete=bool(scan_complete),validation_passed=True,status='complete' if scan_complete else 'partial')
    atomic_json(OUT/'scan_status.json',status)
    atomic_json(OUT/'summary.json',{**status,'matched_ptds_drugs':0,'note':'没有提取到细胞；仅 complete 状态下可判断候选在此版本未观察到'})
    print('没有提取到细胞；先查看 scan_status 和错误表，不把失败当作零交集。')


## 如何使用输出
- 首先确认 `scan_status.json` 的 `complete=true`；partial 结果仅用于试跑/阶段性检查，不代表完整交集。
- `matched_conditions.csv` 是**实际观察到的** PTDS–Tahoe 条件对应表；一个药物可有多个剂量/样本，全部保留。
- `parts_<scope>/*.counts.parquet` 是筛选后单细胞 token/count 原始计数（已去除 CLS）；用 `genes.csv` 的 token_id 对应 Ensembl ID 和 gene_symbol。`.cells.parquet` 仅保存对应元数据。
- `pseudobulk_raw_counts.h5ad` 按样本、细胞系和 plate 求和，不把药物的不同剂量混合。进行 DE 或 logCPM 等分析应另设 notebook，先检查细胞数、批次和对照。
- `drug_resolution_review.csv` 中的失败/歧义需要人工复核；`drug_overrides.csv` 可沿用上一份 notebook 的格式：`ptds_drug,pubchem_cid,reason`。
- `drug_match_evidence.csv` 记录 CID/结构匹配依据；`structure_match_review.csv` 记录骨架相同但尚未确认立体身份的候选，`drug_candidate_coverage.csv` 列出未进入 Tahoe 注释候选的药物。
- Tahoe 与 UniPert 观测层级不同：Tahoe 是单细胞 UMI，UniPert LINCS 是 Level 5 签名，两者不能直接拼接后统一做 counts 归一化。
- 单细胞数量不等于独立干预数量；这里是跨实验条件匹配，不是同一细胞的 RNA–蛋白联合测量。
- 本文件未预填匹配数量。创建时验证了官方真实分片格式、CLS 处理、筛选/聚合逻辑；是否全量完成以你在 Colab 产生的运行状态为准。
